fig_impss — Fig. `fig:impss` (impurity steady state and spin relaxation)
Rows: strongly damped (c3) and weakly damped (c3G0.2) impurity. (a,c) steady-state n_d versus gamma from the empty
impurity; (b,d) M_z^d(t) from the polarized start (IC 10) at gamma = 24.
Reads `data/` (default) or a fresh `output/` (`USE_OUTPUT = True`) and writes `fig_impss.pdf`.

In [1]:
USE_OUTPUT = False
OUTPUT_DIR = "output"

In [2]:
import sys
from pathlib import Path
import numpy as np
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
from common.style import plt, W1, style, panel, tag, legend_handles, LABEL, save
if USE_OUTPUT:
    import postprocess
    D = postprocess.build(OUTPUT_DIR)
else:
    D = IO.read_data(HERE / "data")
ST, MZ = D["steady.csv"], D["mz_curves.csv"]
METHODS = ["HF", "2B", "TPPX", "mGW", "DSL"]
Z = {"exact": 2, "HF": 3, "2B": 3, "TPPX": 3, "mGW": 3, "DSL": 4}
ROWS = [("c3", 24.0, 15, (0.4, 30), "strongly damped"), ("c3G0.2", 24.0, 15, (0.4, 30), "weakly damped")]
def nd(system, m):
    s = IO.select(ST, system=system, method=m, ic="ic00"); s = {k: v[s["gamma"] > 0] for k, v in s.items()}
    o = np.argsort(s["gamma"]); return s["gamma"][o], (s["n_up"] + s["n_dn"])[o]

In [3]:
fig, ax = plt.subplots(2, 2, figsize=(W1, 4.1))
fig.subplots_adjust(left=0.15, right=0.98, top=0.83, bottom=0.10, hspace=0.50, wspace=0.50)
for row, (K, GM, TM, XL, lab) in enumerate(ROWS):
    a = ax[row, 0]
    g, v = nd(K, "exact"); a.plot(g, v, marker="s", ms=2.8, **style("exact"), zorder=2)
    for m in METHODS:
        g, v = nd(K, m); a.plot(g, v, marker="o", ms=2.4, **style(m), zorder=Z[m])
    a.set_xscale("log"); a.set_xlim(*XL); a.set_ylim(0.1, 1.05)
    a.set_xlabel(r"$\gamma$"); a.set_ylabel(r"$n_d$")
    tag(a, lab.replace(" ", "\n"), "ll")
    b = ax[row, 1]
    for m in ["exact"] + METHODS:
        c = IO.select(MZ, system=K, method=m, gamma=GM); b.plot(c["t"], c["Mz"], **style(m), zorder=Z[m])
    b.axhline(0, color="#52514e", lw=0.5); b.set_xlim(0, TM); b.set_ylim(-0.03, 0.52)
    b.set_xlabel(r"$t$"); b.set_ylabel(r"$M_z^d(t)$")
    tag(b, lab + "\n" + rf"$\gamma={GM:g}$", "ur")
for k, a in enumerate(ax.ravel()):
    panel(a, "(%s)" % "abcd"[k])
M = ["exact"] + METHODS
fig.legend(legend_handles(M), [LABEL[m] for m in M], loc="upper center", ncol=3, bbox_to_anchor=(0.56, 1.0))
fig.align_ylabels(ax[:, 0]); fig.align_ylabels(ax[:, 1])
save(fig, HERE / "fig_impss")